# Building A Chatbot - Using Groq

Design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

This chatbot that we build will only use the language model to have a conversation.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  ## aloading all the environment variable

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="Gemma2-9b-It", groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x10712a840>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1047b8230>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage

model.invoke(
    [HumanMessage(content="Hi , My name is Rahul and I am a AI Engineer")]
)


AIMessage(content="Hi Rahul, it's nice to meet you!\n\nThat's awesome that you're an AI Engineer. It's such a fascinating and rapidly evolving field.\n\nWhat kind of AI work are you involved in?  I'm always eager to learn more about the cool things people are doing with AI.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 21, 'total_tokens': 89, 'completion_time': 0.123636364, 'prompt_time': 0.00211448, 'queue_time': 0.069111024, 'total_time': 0.125750844}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--170cd74e-96ba-488e-879a-27fb93dbcdec-0', usage_metadata={'input_tokens': 21, 'output_tokens': 68, 'total_tokens': 89})

In [5]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi , My name is Rahul and I am a AI Engineer"),
        AIMessage(
            content="Hi Rahul, it's nice to meet you!\n\nThat's awesome that you're an AI Engineer. It's such a fascinating and rapidly evolving field.\n\nWhat kind of AI work are you involved in?  I'm always eager to learn more about the cool things people are doing with AI.\n"
        ),
        HumanMessage(content="Hey What's my name and what do I do?"),
    ]
)


AIMessage(content="You told me your name is Rahul, and that you are an AI Engineer!  \n\nIs there anything else you'd like to tell me about yourself or your work? 😊 \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 109, 'total_tokens': 149, 'completion_time': 0.072727273, 'prompt_time': 0.00508333, 'queue_time': 0.06865842700000001, 'total_time': 0.077810603}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--f58cb0a4-6a51-4956-b102-19b17499041e-0', usage_metadata={'input_tokens': 109, 'output_tokens': 40, 'total_tokens': 149})

## Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [7]:
config = {"configurable": {"session_id": "chat1"}}

In [8]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Rahul and I am a AI Engineer")],
    config=config,
)


In [9]:
response.content

"Hello Rahul, it's nice to meet you!  \n\nAs an AI, I don't have personal experiences like being an engineer, but I'm always eager to learn about what you do. \n\nWhat are you currently working on in the field of AI?\n"

In [10]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)


AIMessage(content='Your name is Rahul.  \n\nYou told me at the beginning of our conversation! 😊  \n\n\n\nIs there anything else I can help you with?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 94, 'total_tokens': 127, 'completion_time': 0.06, 'prompt_time': 0.004217861, 'queue_time': 0.084559633, 'total_time': 0.064217861}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--3370a48e-f695-4020-b67b-e8330b62ae80-0', usage_metadata={'input_tokens': 94, 'output_tokens': 33, 'total_tokens': 127})

In [11]:
## change the config-->session id
config1 = {"configurable": {"session_id": "chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="Whats my name")], config=config1
)
response.content


"As an AI, I have no memory of past conversations and don't know your name. If you'd like to tell me, I'm happy to use it! 😊\n"

## Prompt templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all the question to the best of your ability",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model


In [13]:
chain.invoke({"messages": [HumanMessage(content="Hi, my name is Rahul")]})


AIMessage(content="Hello Rahul, it's nice to meet you!  \n\nI'm ready to help with any questions you have.  What can I do for you today? 😊  \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 31, 'total_tokens': 70, 'completion_time': 0.070909091, 'prompt_time': 0.003405871, 'queue_time': 0.09074165899999999, 'total_time': 0.074314962}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--df7d4407-462c-49ad-a7c8-f1689eeaa2a8-0', usage_metadata={'input_tokens': 31, 'output_tokens': 39, 'total_tokens': 70})

In [14]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [15]:
response = chain.invoke(
    {"messages": [HumanMessage(content="Hi My name is Rahul")], "language": "Hindi"}
)
response.content


'नमस्ते राहुल!  मैं आपकी मदद करने के लिए यहाँ हूँ। आपसे क्या पूछना है? 😊 \n'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [16]:
with_message_history = RunnableWithMessageHistory(
    chain, get_session_history, input_messages_key="messages"
)

In [18]:
config = {"configurable": {"session_id": "chat3"}}
repsonse = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi,I am Rahul")], "language": "Hindi"},
    config=config,
)
repsonse.content


'नमस्ते राहुल! 👋 \n\nआप कैसे हैं? 😊 \n'

In [19]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [20]:
response.content

'आपका नाम राहुल है। 😄  \n'

### Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

`trim_messages` helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [21]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)


/Users/rahulsaini/Documents/repositories/gen-ai-cookbook/04-gen-ai/langchain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [24]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What ice cream do i like")],
        "language": "English",
    }
)
response.content

"As a helpful assistant, I don't have access to your personal information, including your ice cream preferences.  \n\nWhat's your favorite flavor?\n"

In [25]:
response = chain.invoke(
    {
        "messages": messages
        + [HumanMessage(content="What was the last addition we did?")],
        "language": "English",
    }
)
response.content

'The last addition we did was 2 + 2, which equals 4.  😄  \n\nIs there anything else I can help you with?\n'

In [26]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config = {"configurable": {"session_id": "chat4"}}

In [27]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content


"As a large language model, I don't have access to past conversations or any personal information about you. So, I don't know your name.\n\nWould you like to tell me? 😊\n"

In [28]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content


"As an AI, I have no memory of past conversations. So, you haven't asked me any math problems before.\n\nWhat math problem can I help you with now? 😊  \n\n"